In [ ]:
import os
import torch
from tqdm import tqdm
from cnet_dataset_sdxl import HSIControlNetDataset   # 你上面给的 Dataset
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)
from warp_and_controlnet_model import WarpAndControlNetModel

# ---------------- 配置 ----------------
ROOT_DIR   = "/data/test"                   # 4 文件夹根目录
CKPT_DIR   = "/data/mv_model_0_with_drop/checkpoint-21500/controlnet/"  # 你的 48 通道 ControlNet 权重
OUT_DIR    = "./results_4in1_sdxl_warp_ckpt_21500_60/"  # 输出目录
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- 数据集 ----------------
dataset = HSIControlNetDataset(root_dir=ROOT_DIR, image_size=512)
if len(dataset) == 0:
    raise RuntimeError("没有 4 组文件夹！")
# sample = dataset[0]                      # 取第一组
# prompt = sample["text"]
# cond48 = torch.from_numpy(sample["conditioning_image"]).unsqueeze(0)  # (1,48,512,512)

# ---------------- 载入 48 通道 ControlNet ----------------
controlnet = WarpAndControlNetModel.from_pretrained(
    CKPT_DIR,
    torch_dtype=torch.float16,
    use_safetensors=True,
    safety_checker=None,
)
controlnet = controlnet.to(torch.float16) 
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "hotshotco/SDXL-512",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)
pipe.enable_model_cpu_offload()

# ---------------- 批量推理 ----------------
# for idx in tqdm(range(len(dataset)), desc="Inference"):
for idx in tqdm(range(3000,3010), desc="Inference"):
    sample = dataset[idx]
    prompt = sample["text"]
    cond48 = sample["conditioning_pixel_values"].unsqueeze(0).half().to(DEVICE)

    image = pipe(
        prompt=prompt,
        # negative_prompt="low quality, blurry",
        image=cond48,
        # num_inference_steps=20,
        # generator=torch.Generator(DEVICE).manual_seed(42),
    ).images[0]

    save_path = os.path.join(OUT_DIR, f"result_{idx:04d}.png")
    image.save(save_path)

print(f"✅ 全部完成，输出已保存到 {OUT_DIR}")

dataset name: /data/test_grid2
[Dataset] 加载缓存...
folders len 240
groups  len 60
Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-21500/controlnet/warper.pt


Inference:   0%|          | 0/60 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/data/test_grid2/embeds/0_prompt_embeds.pt'